# Final Assessment | Cat Detection v2

This notebook documents the Week-1 baseline, the planned Week-2 improvement runs, ONNX export, and the inference checks used for the Docker container.

## 1. Week-1 baseline recap

The Week-1 model was trained in `../m6-04-assessment/runs/detect/cats_n_v1` with `yolo26n.pt`, image size 640, 40 epochs, pretrained weights, default mosaic augmentation, and early stopping patience of 10.

Final validation metrics from `results.csv`:

| Run | Backbone | Tricks | mAP@0.5 | mAP@0.5:0.95 | P | R |
|---|---|---|---:|---:|---:|---:|
| Week-1 baseline | yolo26n | pretrained, mosaic, early stopping | 0.9071 | 0.6920 | 0.8697 | 0.8719 |

Observed weaknesses: small or partially occluded cats are harder to localize, and cluttered backgrounds sometimes create lower-confidence boxes around cat-like texture.

In [ ]:
from pathlib import Path
import pandas as pd

baseline_csv = Path('../m6-04-assessment/runs/detect/cats_n_v1/results.csv')
baseline = pd.read_csv(baseline_csv)
baseline.tail(1)

## 2. Week-2 techniques selected

I selected these techniques for the v2 attempts:

- Larger backbone: try `yolo26s.pt` after the nano baseline, because the baseline is already useful but may underfit small and difficult cats.
- Stronger augmentation: add moderate geometry and color augmentation so the detector sees more pose, scale, lighting, and background variety.
- Longer training with cosine LR: train for 80 epochs with `cos_lr=True` and early stopping to give the larger backbone enough time without overtraining.
- Regularisation: keep weight decay and patience so the model does not chase noisy boxes.

In [ ]:
from ultralytics import YOLO

# Run 1: stronger augmentation on the nano baseline.
model = YOLO('yolo26n.pt')
run1 = model.train(
    data='../m6-04-assessment/data.yaml',
    epochs=60,
    imgsz=640,
    batch=4,
    device='cpu',
    name='cats_v2_aug_n',
    seed=42,
    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.10,
    hsv_h=0.020,
    hsv_s=0.80,
    hsv_v=0.50,
    degrees=5.0,
    translate=0.12,
    scale=0.60,
    fliplr=0.50,
    patience=15,
)

In [ ]:
# Run 2: larger backbone, longer schedule, cosine LR.
model = YOLO('yolo26s.pt')
run2 = model.train(
    data='../m6-04-assessment/data.yaml',
    epochs=80,
    imgsz=640,
    batch=4,
    device='cpu',
    name='cats_v2_s_cos',
    seed=42,
    cos_lr=True,
    lr0=0.006,
    lrf=0.01,
    weight_decay=0.0007,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.05,
    patience=20,
)

In [ ]:
# Run 3: two-stage fine-tuning. First freeze the backbone, then unfreeze with a lower LR.
model = YOLO('yolo26s.pt')
stage1 = model.train(
    data='../m6-04-assessment/data.yaml',
    epochs=10,
    imgsz=640,
    batch=4,
    device='cpu',
    name='cats_v2_s_stage1_head',
    freeze=10,
    lr0=0.003,
    patience=10,
)

model = YOLO('runs/detect/cats_v2_s_stage1_head/weights/best.pt')
stage2 = model.train(
    data='../m6-04-assessment/data.yaml',
    epochs=70,
    imgsz=640,
    batch=4,
    device='cpu',
    name='cats_v2_s_stage2_unfrozen',
    cos_lr=True,
    lr0=0.0015,
    weight_decay=0.0007,
    mosaic=1.0,
    mixup=0.05,
    patience=20,
)

## 3. Comparison table

The currently packaged model is the available Week-1 checkpoint copied into `runs/cats_v2/weights/best.pt` and exported for the container. After running the v2 training cells above, replace the v2 metric placeholders with the best run's `results.csv` values and point the export cell to that run.

| Run | Backbone | Tricks | mAP@0.5 | mAP@0.5:0.95 | P | R |
|---|---|---|---:|---:|---:|---:|
| Week-1 baseline | yolo26n | pretrained, mosaic, early stopping | 0.9071 | 0.6920 | 0.8697 | 0.8719 |
| v2 - run 1 | yolo26n | stronger augmentation | pending | pending | pending | pending |
| v2 - run 2 | yolo26s | larger backbone, cosine LR, regularisation | pending | pending | pending | pending |
| v2 - run 3 | yolo26s | two-stage fine-tune, cosine LR, regularisation | pending | pending | pending | pending |
| **v2 - shipped** | yolo26n | exported ONNX container baseline | 0.9071 | 0.6920 | 0.8697 | 0.8719 |

## 4. Export to ONNX

In [ ]:
from ultralytics import YOLO

model = YOLO('runs/cats_v2/weights/best.pt')
onnx_path = model.export(format='onnx', imgsz=640, opset=17, dynamic=False)
onnx_path

## 5. ONNX sanity check

The exported model should return `(1, 300, 6)` detections in `[xmin, ymin, xmax, ymax, confidence, class]` format. The check below compares the top detections from PyTorch and ONNX on a few test images.

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image

def letterbox(img, imgsz=640):
    w, h = img.size
    scale = min(imgsz / w, imgsz / h)
    nw, nh = round(w * scale), round(h * scale)
    pad_x, pad_y = (imgsz - nw) / 2, (imgsz - nh) / 2
    canvas = Image.new('RGB', (imgsz, imgsz), (114, 114, 114))
    canvas.paste(img.resize((nw, nh)), (round(pad_x - 0.1), round(pad_y - 0.1)))
    return canvas

session = ort.InferenceSession('runs/cats_v2/weights/best.onnx', providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
print('ONNX output shape:', session.get_outputs()[0].shape)

test_images = sorted(Path('../m6-04-assessment/data/DATA_CLEAN/images').glob('*.jpg'))[:5]
for image_path in test_images:
    img = Image.open(image_path).convert('RGB')
    arr = (np.asarray(letterbox(img), dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]
    onnx_out = session.run(None, {input_name: arr})[0][0]
    torch_result = model.predict(str(image_path), imgsz=640, conf=0.25, verbose=False)[0]
    print(image_path.name, 'onnx detections >= .25:', int((onnx_out[:, 4] >= 0.25).sum()), 'torch boxes:', len(torch_result.boxes))